In [ ]:
from xgboost import XGBClassifier

In [ ]:
# !pip install groq

In [ ]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()   # load .env file

api_key = os.getenv("API_GROK_KEY")

client = Groq(api_key=api_key)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle

In [ ]:
df = pd.read_csv("student_data.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df = df[['studytime', 'failures', 'absences', 'G1', 'G2', 'G3', 'goout']]

In [ ]:
def categorize_risk(x):
    if x < 8:
        return 2   # High Risk
    elif x < 12:
        return 1   # Medium Risk
    else:
        return 0

df['risk'] = df['G3'].apply(categorize_risk)

In [ ]:
df = df.drop(columns=['G3'])


In [ ]:
X = df[['studytime', 'failures', 'absences', 'G1', 'G2']]
y = df['risk']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# test commit
model = XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
import numpy as np

y_pred = model.predict(X_test)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", accuracy)

In [ ]:
def generate_explanation(student_row):
    reasons = []

    if student_row['G1'] < 10:
        reasons.append("Low early academic performance (G1)")

    if student_row['absences'] > 10:
        reasons.append("High absenteeism")

    if student_row['failures'] > 0:
        reasons.append("History of academic failures")

    if student_row['studytime'] <= 1:
        reasons.append("Low study time")

    if student_row['goout'] >= 4:
        reasons.append("High social activity affecting academics")

    return reasons


In [ ]:
input_data = df.iloc[0]

In [ ]:
reasons = generate_explanation(input_data)
print("Identified Risk Factors:")
for r in reasons:
    print("-", r)

In [ ]:
risk_labels = {
    0: "Low Risk",
    1: "Medium Risk",
    2: "High Risk"
}

for i in range(10):
    print("Student", i+1)
    print("Study Time:", X_test.iloc[i]['studytime'])
    print("Failures:", X_test.iloc[i]['failures'])
    print("Absences:", X_test.iloc[i]['absences'])
    print("G1:", X_test.iloc[i]['G1'])
    print("G2:", X_test.iloc[i]['G2'])
    print("Predicted Risk:", risk_labels[y_pred[i]])
    print("-" * 30)

# for i in range(10):
    # print("Student", i+1)
    # print("Study Time:", X_test.iloc[i]['studytime'])
    # print("Failures:", X_test.iloc[i]['failures'])
    # print("Absences:", X_test.iloc[i]['absences'])
    # print("G1:", X_test.iloc[i]['G1'])
    # print("G2:", X_test.iloc[i]['G2'])
    # print("Predicted Risk:", categorize_risk(y_pred[i]))
    # print("-" * 30)

In [ ]:
def generate_summary(reasons):
    if not reasons:
        return "Student shows moderate performance indicators. Monitor regularly."

    summary = "The student is at academic risk due to: "
    summary += ", ".join(reasons)
    summary += ". Early academic intervention is recommended."

    return summary

print(generate_summary(reasons))

In [ ]:
def genai_student_report(name, risk_level, marks, absences, failures, study_hours):

    from dotenv import load_dotenv
    import os
    from groq import Groq

    load_dotenv()
    client = Groq(api_key=os.getenv("API_GROK_KEY"))

    prompt = f"""
    You are an AI education assistant.

    Student Name: {name}
    Risk Level: {risk_level}
    Marks: {marks}
    Absences: {absences}
    Failures: {failures}
    Study Hours per day: {study_hours}

    Generate output in 3 sections:
    1. Personalized Learning Plan
    2. Teacher Report
    3. Parent Feedback Message

    Keep language simple and clear.
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
prediction = "High Risk"
name = "riya"

output = genai_student_report(name,prediction, marks=45, absences=12, failures=3,study_hours = 8)
print(output)

In [ ]:
# pickle.dump(model, open("student_model.pkl", "wb"))
# print("\nModel saved as student_model.pkl")

import joblib

joblib.dump(model, "student_model.pkl")
print("\nModel saved as student_model.pkl")

In [ ]:
import joblib
joblib.dump(model, "student_model.pkl")